# Train and evaluate on a free GPU (Colab or Kaggle)

This notebook runs the full experiment:

1. Download the pinned dataset and build the grouped (leakage-reduced) split.
2. Train YOLOv8n twice: on the grouped split (main result) and on the official RF100 split (for comparison).
3. Evaluate both on their own test sets, sweep confidence thresholds on validation data, and draw figures.

**Colab:** Runtime → Change runtime type → **T4 GPU**, then run the cells in order. Results are saved to Google Drive in `MyDrive/asbestos_runs`, so a disconnect does not lose finished epochs.

**Kaggle:** in the notebook settings, choose a **GPU T4** accelerator and turn **Internet on**, then use **Save Version → Save & Run All** to run everything in the background. Results are saved in the notebook's Output under `asbestos_runs`.

Each training run takes roughly 30–60 minutes on a T4; the time per epoch in the training log gives a better estimate.

In [1]:
!nvidia-smi -L  # should list a GPU

GPU 0: Tesla T4 (UUID: GPU-896589dd-e0d5-05c6-aa20-c40cf18b735f)


## 1. Choose where results are saved

In [3]:
import os

ON_KAGGLE = os.path.isdir("/kaggle/working")
if ON_KAGGLE:  # results become the notebook's output; code and data stay in temporary space
    WORK, RUNS = "/kaggle/temp", "/kaggle/working/asbestos_runs"
else:  # Colab: results go to Google Drive and survive disconnects
    from google.colab import drive

    drive.mount("/content/drive")
    WORK, RUNS = "/content", "/content/drive/MyDrive/asbestos_runs"
WORKERS = min(4, os.cpu_count() or 2)  # data-loading processes: Colab has 2 CPUs, Kaggle 4
os.makedirs(WORK, exist_ok=True)
os.makedirs(RUNS, exist_ok=True)
print("Results will be saved in", RUNS)

Mounted at /content/drive
Results will be saved in /content/drive/MyDrive/asbestos_runs


## 2. Get the code and install it

Red dependency-conflict warnings from pip are expected here and do not affect this project.

In [4]:
REPO = f"{WORK}/asbestos-fiber-detection"
![ -d {REPO} ] || git clone -q https://github.com/sp-ankitprajapati/asbestos-fiber-detection {REPO}
%cd {REPO}
!pip install -q -r requirements.txt

/content/asbestos-fiber-detection


## 3. Download the data and build the grouped split

`prepare_data.py` downloads a fixed revision of the dataset and re-runs the audit.
`make_grouped_split.py` keeps images that share a source name (or identical bytes) in the same split.
The grouping is deterministic (seed 42), so re-running this cell after a disconnect rebuilds the same split.
If the download stops with a rate-limit (429) error, wait a few minutes and run the cell again; finished files are kept.

In [5]:
!python prepare_data.py
![ -f asbestos/grouped/data.yaml ] || python make_grouped_split.py

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2667 files:   0% 0/2667 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/61.0k [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/61.0k [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/96.9k [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/99.4k [00:00<?, ?B/s]

Fetching 2667 files:   0% 1/2667 [00:00<20:21,  2.18it/s]
Reconstructing (incomplete total...):   2% 2.46k/127k [00:00<00:23, 5.38kB/s] 
Reconstructing (incomplete total...):   2% 2.46k/130k [00:00<00:23, 5.38kB/s]
Reconstructing (incomplete total...):   3% 5.38k/164k [00:00<00:29, 5.38kB/s]
Reconstructing (incomplete total...):   3% 5.38k/165k [00:00<00:29, 5.38kB/s]
Reconstructing (incomplete total...):   3% 6.02k/197k [00:00<00:35, 5.3

## 4. Train on the grouped split (main model)

Fine-tunes COCO-pretrained YOLOv8n for up to 100 epochs. Training stops early if the validation score has not improved for 20 epochs.

In [6]:
!python train.py --data asbestos/grouped/data.yaml --name grouped --project {RUNS} --output {RUNS}/grouped.pt --device 0 --workers {WORKERS}

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
New https://pypi.org/project/ultralytics/8.4.207 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/asbestos-fiber-detection/asbestos/grouped/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropou

**Colab only, if it disconnected during training:** run sections 1–3 again, then remove the `#` from the matching line below and run it to continue from the last saved epoch. If you started a run more than once, use the newest folder (for example `grouped2`). Kaggle's Save & Run All does not need this.

In [ ]:
# !python train.py --resume {RUNS}/grouped/weights/last.pt --output {RUNS}/grouped.pt --device 0 --workers {WORKERS}
# !python train.py --resume {RUNS}/official/weights/last.pt --output {RUNS}/official.pt --device 0 --workers {WORKERS}

## 5. Train on the official RF100 split (for comparison)

Same settings, but on the official split, where variants of the same source image sit in both train and test.
Comparing the two test scores shows how much the official split may overstate performance.

In [7]:
!python train.py --data asbestos/data.local.yaml --name official --project {RUNS} --output {RUNS}/official.pt --device 0 --workers {WORKERS}

New https://pypi.org/project/ultralytics/8.4.207 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/asbestos-fiber-detection/asbestos/data.local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train

## 6. Evaluate

- **Validation first:** standard mAP, plus a confidence-threshold sweep for choosing a review setting.
- **Then each model on its own split's test set, once.** The two test sets contain different images, so treat a gap as a warning sign, not an exact measure of leakage.

In [8]:
!mkdir -p reports/results
!python evaluate.py --weights {RUNS}/grouped.pt --data asbestos/grouped/data.yaml --split val --output reports/results/grouped_val.json --device 0
!python threshold_sweep.py --weights {RUNS}/grouped.pt --data asbestos/grouped/data.yaml --output reports/results/grouped_val_thresholds.csv --device 0
!python evaluate.py --weights {RUNS}/grouped.pt --data asbestos/grouped/data.yaml --split test --output reports/results/grouped_test.json --device 0
!python evaluate.py --weights {RUNS}/official.pt --data asbestos/data.local.yaml --split test --output reports/results/official_test.json --device 0

Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,436 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1657.3±392.2 MB/s, size: 31.0 KB)
val: Scanning /content/asbestos-fiber-detection/asbestos/grouped/val/labels.cache... 273 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 273/273 49.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 10.0it/s 3.5s
                   all        273       1912      0.661      0.555        0.6      0.407
       thick-dark-mark         59        251      0.789      0.785      0.819      0.577
      thick-light-mark        264       1238      0.732       0.57      0.646        0.4
        thin-dark-mark        101        206      0.683      0.505      0.585      0.443
       thin-light-mark        126        217      0.439      0.359       0.35      0.209
Speed: 0.4ms pr

In [9]:
import json

for name in ["grouped_val", "grouped_test", "official_test"]:
    result = json.load(open(f"reports/results/{name}.json"))
    print(f"{name:14s} mAP50 {result['map50']:.3f}   mAP50-95 {result['map50_95']:.3f}")
    for row in result["per_class"]:
        print(f"    {row['class']:17s} AP50 {row['map50']:.3f}")

grouped_val    mAP50 0.600   mAP50-95 0.407
    thick-dark-mark   AP50 0.819
    thick-light-mark  AP50 0.646
    thin-dark-mark    AP50 0.585
    thin-light-mark   AP50 0.350
grouped_test   mAP50 0.495   mAP50-95 0.328
    thick-dark-mark   AP50 0.769
    thick-light-mark  AP50 0.639
    thin-dark-mark    AP50 0.270
    thin-light-mark   AP50 0.303
official_test  mAP50 0.622   mAP50-95 0.406
    thick-dark-mark   AP50 0.850
    thick-light-mark  AP50 0.671
    thin-dark-mark    AP50 0.631
    thin-light-mark   AP50 0.337


## 7. Figures

`make_figures.py` redraws the README charts and adds the per-class AP comparison.
The sample image shows the grouped model on the first three grouped test images, sorted by file name (not hand-picked), at confidence 0.25.

In [10]:
!python make_figures.py

Saved assets/split_overlap.png
Saved assets/split_overlap_dark.png
Saved assets/class_balance.png
Saved assets/class_balance_dark.png
Saved assets/per_class_ap.png
Saved assets/per_class_ap_dark.png


In [11]:
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

model = YOLO(f"{RUNS}/grouped.pt")
paths = sorted(Path("asbestos/grouped/test/images").glob("*.jpg"))[:3]
plots = [r.plot(line_width=2) for r in model.predict([str(p) for p in paths], conf=0.25, verbose=False)]
tiles = [cv2.resize(im, (round(im.shape[1] * 480 / im.shape[0]), 480)) for im in plots]
cv2.imwrite("assets/sample_predictions.jpg", np.hstack(tiles))  # plot() and imwrite both use BGR

True

## 8. Collect the results

Copies everything worth committing into `asbestos_runs/to_commit`, laid out like the repository, and packs it as `asbestos_runs/asbestos_results.zip`. Checkpoints (`.pt`) are not included. On Colab these are in Google Drive; on Kaggle they are in the notebook's Output.
Add the files to the repository, then fill in the Results table in `README.md`.

In [12]:
import shutil

!cp {RUNS}/grouped.metadata.json {RUNS}/official.metadata.json reports/results/
!mkdir -p {RUNS}/to_commit/reports
!cp -r reports/results {RUNS}/to_commit/reports/
!cp -r assets {RUNS}/to_commit/
shutil.make_archive(f"{RUNS}/asbestos_results", "zip", RUNS, "to_commit")
!ls -R {RUNS}/to_commit

/content/drive/MyDrive/asbestos_runs/to_commit:
assets	reports

/content/drive/MyDrive/asbestos_runs/to_commit/assets:
class_balance_dark.png	per_class_ap.png	split_overlap.png
class_balance.png	sample_predictions.jpg
per_class_ap_dark.png	split_overlap_dark.png

/content/drive/MyDrive/asbestos_runs/to_commit/reports:
results

/content/drive/MyDrive/asbestos_runs/to_commit/reports/results:
grouped.metadata.json  grouped_val_thresholds.csv	     official_test.json
grouped_test.json      grouped_val_thresholds.metadata.json
grouped_val.json       official.metadata.json
